# GRPO Training — Clinical Data Cross-Domain Validation (Task 4)

This notebook fine-tunes a 7B LLM using GRPO (Group Relative Policy Optimization) on Task 4 of the Clinical Data Standardization environment.

**Baseline (zero-shot llama-3.3-70b-versatile):** 0.4388  
**Target:** 0.65+ on the pharmaverse Task 4 benchmark

**Hardware required:** Kaggle P100 or T4 (16GB VRAM)

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install -q trl>=0.12.0 transformers>=4.47.0 peft>=0.14.0 bitsandbytes>=0.45.0 accelerate>=1.2.0 datasets huggingface_hub


## Cell 2 — Clone repo and import grader

In [ ]:
import subprocess, sys, json, os

# Clone the environment repo
if not os.path.exists('/kaggle/working/clinical-data-env'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/thisis-gp/clinical-data-env.git',
         '/kaggle/working/clinical-data-env'],
        check=True
    )

# Add grader to path
REPO = '/kaggle/working/clinical-data-env'
sys.path.insert(0, REPO)

from graders.grader_task4 import grade_task4

print('Grader imported successfully')

## Cell 3 — Load Task 4 cases and build dataset

In [ ]:
import json
from datasets import Dataset

SYSTEM_PROMPT = """You are a clinical data standardization expert trained in CDISC standards.

You will receive cross-domain SDTM records (DM, EX, CM, AE, DS) and must identify any inconsistencies.
You must respond with ONLY a valid JSON object — no markdown, no explanation outside the JSON.

Your response must have this exact structure:
{"issues": [{"type": "...", "domain": "...", "field": "...", "description": "..."}]}

If no cross-domain inconsistencies exist, return: {"issues": []}

Use these canonical issue types:
  - "dm_ex_date_mismatch" — DM.RFSTDTC does not match earliest EX.EXSTDTC
  - "prohibited_cm_before_first_dose" — prohibited CM starts before RFSTDTC
  - "orphan_sae" — AE with AESER='Y' has no matching DS record

Return ONLY the JSON object."""


def load_cases(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)['cases']


def case_to_prompt(case):
    """Convert a task4 case to a chat prompt."""
    user_content = (
        f"STUDY CONTEXT & RULES:\n{case['study_context']}\n\n"
        f"TASK:\n{case['task_description']}\n\n"
        f"INPUT DATA:\n{json.dumps(case['input_data'], indent=2)}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


# Load toy + pharmaverse cases for training
toy_cases = load_cases(f'{REPO}/data/task4_cases.json')
pharmaverse_cases = load_cases(f'{REPO}/data/pharmaverse/task4_cases.json')
all_cases = toy_cases + pharmaverse_cases

print(f'Total training cases: {len(all_cases)} ({len(toy_cases)} toy + {len(pharmaverse_cases)} pharmaverse)')

# Build HuggingFace dataset — store ground truth in extra column for reward fn
records = [
    {
        'prompt': case_to_prompt(case),
        'ground_truth': json.dumps(case['ground_truth']),
        'case_id': case['case_id'],
        'difficulty': case.get('difficulty', 'unknown'),
    }
    for case in all_cases
]

dataset = Dataset.from_list(records)
print(dataset)

## Cell 4 — Define reward function

In [ ]:
import re


def extract_json(text: str) -> dict | None:
    """Extract JSON from model output, tolerating markdown code blocks."""
    text = text.strip()

    # Strip markdown code fences
    if text.startswith('```'):
        text = re.sub(r'^```[\w]*\n?', '', text)
        text = re.sub(r'\n?```$', '', text).strip()

    # Try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to extract first JSON object
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    return None


def reward_fn(completions, ground_truth, **kwargs) -> list[float]:
    """
    GRPO reward function.
    completions: list of model output strings
    ground_truth: list of JSON strings (one per prompt in the batch)
    Returns list of float scores in (0, 1).
    """
    scores = []
    for completion, gt_str in zip(completions, ground_truth):
        gt = json.loads(gt_str)
        parsed = extract_json(completion)

        if parsed is None:
            scores.append(0.01)  # unparseable output
            continue

        # grade_task4 expects output_data to have an 'issues' key
        # If the model returned {"issues": [...]} directly, use as-is
        if 'issues' not in parsed and 'output_data' in parsed:
            parsed = parsed['output_data']

        score, _, _ = grade_task4(parsed, gt)
        scores.append(float(score))

    return scores


# Quick sanity check
test_gt = json.dumps({"issues": [{"type": "dm_ex_date_mismatch", "domain": "DM/EX", "field": "RFSTDTC/EXSTDTC", "description": "dates do not match"}]})
test_completions = [
    '{"issues": [{"type": "dm_ex_date_mismatch", "domain": "DM/EX", "field": "RFSTDTC/EXSTDTC", "description": "DM RFSTDTC does not match EX EXSTDTC"}]}',  # correct
    '{"issues": []}',   # missed
    'not json at all',  # parse error
]
test_scores = reward_fn(test_completions, [test_gt] * 3)
print(f'Reward sanity check: {test_scores}')  # expect [~0.99, low, 0.01]

## Cell 5 — Baseline evaluation (before training)

In [ ]:
import os
from pathlib import Path

import torch
from huggingface_hub import login, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
HF_CACHE_DIR = Path('/kaggle/working/hf-cache')
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle-friendly Hugging Face cache settings.
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('Loaded HF token from Kaggle secrets.')
except Exception:
    print('No Kaggle HF_TOKEN secret found. Continuing without explicit login.')

if not torch.cuda.is_available():
    raise RuntimeError('This notebook expects a GPU runtime on Kaggle.')

supports_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if supports_bf16 else torch.float16
dtype_name = 'bfloat16' if supports_bf16 else 'float16'

print(f'Preparing local snapshot for {MODEL_ID}...')
print('If this hangs on Kaggle, make sure Internet is enabled and the model is accessible from your HF account.')

local_model_dir = snapshot_download(
    repo_id=MODEL_ID,
    cache_dir=str(HF_CACHE_DIR),
    resume_download=True,
    local_dir=str(HF_CACHE_DIR / 'models' / MODEL_ID.replace('/', '--')),
    local_dir_use_symlinks=False,
    max_workers=2,
    ignore_patterns=['*.onnx', '*.tflite', '*.gguf', '*.md'],
)

print(f'Loading {MODEL_ID} with 4-bit quantization using {dtype_name}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(local_model_dir, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    local_files_only=True,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=compute_dtype,
)

print(f'Model loaded from {local_model_dir}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')


In [ ]:
def run_eval(model, tokenizer, cases, label='eval'):
    """Run greedy inference on all cases and return mean score."""
    model.eval()
    scores_by_difficulty = {}
    all_scores = []

    for case in cases:
        prompt = case_to_prompt(case)
        text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        completion = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        gt_str = json.dumps(case['ground_truth'])
        score = reward_fn([completion], [gt_str])[0]

        difficulty = case.get('difficulty', 'unknown')
        scores_by_difficulty.setdefault(difficulty, []).append(score)
        all_scores.append(score)

        print(f"  [{label}] {case['case_id']} ({difficulty}): {score:.3f}")

    mean = sum(all_scores) / len(all_scores) if all_scores else 0.0
    print(f'\n[{label}] Overall: {mean:.4f}')
    for diff, s in sorted(scores_by_difficulty.items()):
        print(f'  {diff}: {sum(s)/len(s):.4f} ({len(s)} cases)')
    return mean, scores_by_difficulty, all_scores


# Evaluate on pharmaverse cases only (same split as our published baseline)
print('=== BASELINE (before training) ===')
baseline_mean, baseline_by_diff, baseline_scores = run_eval(
    base_model, tokenizer, pharmaverse_cases, label='baseline'
)

## Cell 6 — Configure and run GRPO training

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import GRPOConfig, GRPOTrainer

# Prepare model for LoRA fine-tuning
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

grpo_config = GRPOConfig(
    # Training
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    bf16=True,

    # GRPO specific
    num_generations=4,        # sample 4 completions per prompt, compare within group
    max_new_tokens=512,
    temperature=0.9,          # high temp during training for diversity
    top_p=0.9,

    # Logging
    logging_steps=5,
    output_dir='/kaggle/working/grpo-task4',
    save_strategy='epoch',
    report_to='none',

    # Memory
    max_prompt_length=1024,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
)

trainer = GRPOTrainer(
    model=peft_model,
    reward_funcs=reward_fn,
    args=grpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print(f'Training on {len(dataset)} cases × {grpo_config.num_generations} generations × {grpo_config.num_train_epochs} epochs')
print('Starting GRPO training...')
trainer.train()

## Cell 7 — Post-training evaluation

In [ ]:
print('=== POST-TRAINING (after GRPO) ===')
trained_mean, trained_by_diff, trained_scores = run_eval(
    peft_model, tokenizer, pharmaverse_cases, label='trained'
)

## Cell 8 — Results comparison

In [ ]:
print('\n' + '='*60)
print('RESULTS: Task 4 Cross-Domain Validation')
print('='*60)
print(f'  Model:    {MODEL_ID} + LoRA GRPO fine-tune')
print(f'  Dataset:  {len(pharmaverse_cases)} pharmaverse cases')
print(f'  Epochs:   {grpo_config.num_train_epochs}')
print()
print(f'  {"Metric":<25} {"Baseline":>10} {"Trained":>10} {"Delta":>10}')
print(f'  {"-"*55}')
print(f'  {"Overall mean":<25} {baseline_mean:>10.4f} {trained_mean:>10.4f} {trained_mean - baseline_mean:>+10.4f}')

all_diffs = sorted(set(list(baseline_by_diff.keys()) + list(trained_by_diff.keys())))
for diff in all_diffs:
    b = sum(baseline_by_diff.get(diff, [0])) / max(len(baseline_by_diff.get(diff, [1])), 1)
    t = sum(trained_by_diff.get(diff, [0])) / max(len(trained_by_diff.get(diff, [1])), 1)
    print(f'  {diff:<25} {b:>10.4f} {t:>10.4f} {t - b:>+10.4f}')

print('='*60)
improvement = trained_mean - baseline_mean
if improvement > 0:
    print(f'\nImprovement: +{improvement:.4f} ({improvement/baseline_mean*100:.1f}% relative)')
else:
    print(f'\nNo improvement this run. Try more epochs or higher learning rate.')

## Cell 9 — Save adapter and push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

# Set your HuggingFace token here or via Kaggle secrets
HF_TOKEN = os.getenv('HF_TOKEN', '')  # add as Kaggle secret: Settings → Add-ons → Secrets
HF_REPO_ID = 'thisis-gp/clinical-data-grpo-task4'  # will be created if it doesn't exist

if HF_TOKEN:
    peft_model.save_pretrained('/kaggle/working/grpo-task4-adapter')
    tokenizer.save_pretrained('/kaggle/working/grpo-task4-adapter')

    peft_model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print(f'Adapter pushed to https://huggingface.co/{HF_REPO_ID}')
else:
    # Save locally if no HF token
    peft_model.save_pretrained('/kaggle/working/grpo-task4-adapter')
    tokenizer.save_pretrained('/kaggle/working/grpo-task4-adapter')
    print('Adapter saved to /kaggle/working/grpo-task4-adapter')
    print('Set HF_TOKEN in Kaggle Secrets to push to HuggingFace Hub.')

## Cell 10 — Generate README results snippet

In [ ]:
# Copy-paste this into the README Training Results section
snippet = f"""## RL Training Results

Model: `{MODEL_ID}` fine-tuned with GRPO on Task 4 (cross-domain validation).

| Metric | Baseline (zero-shot) | After GRPO | Delta |
|--------|---------------------|------------|-------|
| Task 4 overall | {baseline_mean:.4f} | {trained_mean:.4f} | {trained_mean - baseline_mean:+.4f} |
"""
for diff in all_diffs:
    b = sum(baseline_by_diff.get(diff, [0])) / max(len(baseline_by_diff.get(diff, [1])), 1)
    t = sum(trained_by_diff.get(diff, [0])) / max(len(trained_by_diff.get(diff, [1])), 1)
    snippet += f'| Task 4 {diff} | {b:.4f} | {t:.4f} | {t-b:+.4f} |\n'

snippet += f"""
Training config: LoRA r=16, GRPO {grpo_config.num_generations} generations, \
{grpo_config.num_train_epochs} epochs, lr={grpo_config.learning_rate}
"""
print(snippet)